## **Fine-Tuning mBERT for Sentiment Analysis**
Created by Genevieve Powell

---

Sources:

*   FineTuning BERT for Arabic Sentiment Analysis: https://colab.research.google.com/github/khaledsoudy-1/FineTuning-BERT-Arabic-Sentiment/blob/main/Fine_Tuning_BERT.ipynb#scrollTo=1wVm7ZxozkRG
*   Recent FineTuning application for BERT: https://dev.to/nareshnishad/fine-tuning-bert-unlocking-the-power-of-pre-trained-language-models-4i1k


In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 12.3 MB/s  0:00:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1


In [2]:
#importing mBERT model for sentiment analysis
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, ignore_mismatched_sizes=True)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlptown/bert-base-multilingual-uncased-sentiment
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([5]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([5, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#import TUNIZI Data from Google Drive
file_path = "/content/drive/MyDrive/CompNeuro Project/TUNIZI_V2.txt"

In [5]:
#Creating df from data file
import pandas as pd

df = pd.read_csv(file_path, sep=';', header=None, names=["text", "label"])
df

,text,label
0,7yeti ca ya slouma,1
1,rabi yehdik aka chenajmou n9oulo,1
2,amazing like usual,1
3,bonne chance docteur marzouki vive cpr,1
4,allah yarhmou we rabi saber ahlo,1
...,...,...
99995,win wsilt fil régime,0
99996,chenouwa ili rachitou ala el pinceau 3aychik,0
99997,astuce blush molka hedha,0
99998,chnowa ismou,0


In [6]:
#Generating five random lines, random_state=41 means the same five will be chosen each time
random_5_rows = df.sample(n=5, random_state=41)
random_5_rows

,text,label
48283,ohhh hhhhhhhhhhhh walh bdmou3,1
99865,ok khawla,0
96557,winou ryuk,0
10456,ooo wlh m3alem 9wiya yasser,1
75754,fama 5edma brabi rani batal,-1


In [7]:
#Copy five random rows to avoid modifying the original dataset
five_rows_copy = random_5_rows.copy()

#BERT sentiment models automatically have five sentiment categories, map to three (Positive, Neutral, Negative)
custom_mapping = {
    0: -1,
    1: -1,
    2: 0,
    3: 1,
    4: 1
}

In [8]:
#Testing the mBERT model on the dataset
import numpy as np
import torch

predicted_sentiments_list = []

for row in five_rows_copy['text']:
  inputs = tokenizer(row, return_tensors="pt")

  #Move inputs to the same device as the model
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  #Run model inference
  with torch.no_grad():
    outputs = model(**inputs)

  #Get logits and move to CPU
  logits = outputs.logits.detach().cpu().numpy()

  #Find the predicted class index
  predicted_class = np.argmax(logits, axis=-1)[0]

  #Map to custom sentiment label (handling out of bounds index if needed)
  predicted_sentiment = custom_mapping.get(predicted_class, predicted_class)

  #Append to sentiments list
  predicted_sentiments_list.append(predicted_sentiment)

print(predicted_sentiments_list)

[-1, 0, 0, 0, -1]


In [9]:
#Comparing the random sample labels to the sentiment predictions before fine-tuning
five_rows_copy['Bert_Sentiments'] = predicted_sentiments_list
five_rows_copy

,text,label,Bert_Sentiments
48283,ohhh hhhhhhhhhhhh walh bdmou3,1,-1
99865,ok khawla,0,0
96557,winou ryuk,0,0
10456,ooo wlh m3alem 9wiya yasser,1,0
75754,fama 5edma brabi rani batal,-1,-1


In [10]:
df.shape

(100000, 2)

In [11]:
#Randomly sample 30000 out of 10000 TUNIZI comments, for shorter runtime
df = df.sample(n=30000, random_state=42)

print(f"Total Number of Rows and Columns After Sampling: {df.shape}")

Total Number of Rows and Columns After Sampling: (30000, 2)


In [12]:
#Print unique labels in the dataset
import numpy as np

for label in np.unique(df['label']):
  print(label)

-1
0
1


In [13]:
#Changes -1:Negative, 0:Neutral, 1:Positive labels to sequential 0, 1, 2 sentiment labels
from sklearn.preprocessing import LabelEncoder

LE = LabelEncoder()

df['label'] = LE.fit_transform(df['label'])
df.head()

,text,label
75721,fare5 lgadhefi hhhhh,0
80184,ana en7eb ca ama nakreh esliem rya7e 5atro tab...,0
19864,ne9sa djabou,2
76699,mskina tchaf,0
92991,mella 3lech mabathouch el barnemj wenty madfa3...,0


In [14]:
#Check after converting labels into numerics
for label in np.unique(df['label']):
  print(label)

0
1
2


In [15]:
#Splits training dataset into training and validation
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, test_size=0.2, random_state=42)

In [16]:
#Check the new shapes
print(train.shape)
print(val.shape)

(24000, 2)
(6000, 2)


In [17]:
#Checking sets
train.head()

,text,label
30092,bl7a9 bakiitni athrtt fiya barcha raby yfadhll...,2
66269,fammech 7wija 3la ettarachi hhhhhh,0
88117,saib 3lik mazilt b3id barcha ija bahdaya tawa ...,0
34170,c est un sujet tres interessant ken je avec ra...,2
75794,9ahba haifa bouli,0


In [18]:
#Reseting the index values
train.reset_index(drop=True, inplace=True)
val.reset_index(drop=True, inplace=True)

train.head()

,text,label
0,bl7a9 bakiitni athrtt fiya barcha raby yfadhll...,2
1,fammech 7wija 3la ettarachi hhhhhh,0
2,saib 3lik mazilt b3id barcha ija bahdaya tawa ...,0
3,c est un sujet tres interessant ken je avec ra...,2
4,9ahba haifa bouli,0


In [19]:
train.head()

,text,label
0,bl7a9 bakiitni athrtt fiya barcha raby yfadhll...,2
1,fammech 7wija 3la ettarachi hhhhhh,0
2,saib 3lik mazilt b3id barcha ija bahdaya tawa ...,0
3,c est un sujet tres interessant ken je avec ra...,2
4,9ahba haifa bouli,0


In [20]:
val.head()

,text,label
0,degage sayebha jam3iya enta w jma3tk degage de...,0
1,hhhhhhhhh ama eti7ed ta7ki 3lih brabi mta3 eso...,0
2,brobi wine kane slim fi 3ahde ben ali,2
3,dz alger mayna likm wlah,0
4,w niktelek bouk enty zeda barra imchy nayek fr...,0


In [21]:
#Converts text and label columns of the traing data into Numpy arrays
x_train = train['text'].to_numpy()
y_train = train['label'].to_numpy()

#Apply to the validation data
x_test = val['text'].to_numpy()
y_test = val['label'].to_numpy()

In [22]:
print(f"Text column of the Train Data:\n {x_train}")
print(f"\nLabel column of the Train Data:\n {y_train}")
print(type(x_train))
print(type(y_train))

Text column of the Train Data:
 ['bl7a9 bakiitni athrtt fiya barcha raby yfadhllk omk y 5ouya nchalah lebess 9lil bch tal9a hkeya kima haka raby y5alik o bravo'
 'fammech 7wija 3la ettarachi hhhhhh'
 'saib 3lik mazilt b3id barcha ija bahdaya tawa infahmik ishih' ...
 'barcha zawdin' 'lik wahra ya si lmarzou9i mouch 3adiya hhhhhhhhhhhhh'
 'enchalah labés ya slayem']

Label column of the Train Data:
 [2 0 0 ... 1 2 2]
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [23]:
print(f"Text column of the Validation (Test) Data:\n {x_test}")
print(f"\nLabel column of the Validation (Test) Data:\n {y_test}")

Text column of the Validation (Test) Data:
 ['degage sayebha jam3iya enta w jma3tk degage degage goub3a riahi riahi sallami'
 'hhhhhhhhh ama eti7ed ta7ki 3lih brabi mta3 esora9 wel 5awana w eke el3abasi elkalb tawa yjih nhar'
 'brobi wine kane slim fi 3ahde ben ali' ... 'tmout abche3 ilmoutet'
 'hhh ma3andek chay ken lbas' 'enchalh rabi m3ak']

Label column of the Validation (Test) Data:
 [0 0 2 ... 0 0 2]


In [24]:
from torch.utils.data import Dataset
import torch

#Defines a class to hold the dataset and tokenizer
class SentimentDataset(Dataset):

    #Initialization step
    def __init__(self, texts, labels, tokenizer):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer

    #Returns length of sentiment text lengths
    def __len__(self):
        return len(self.texts)

    #Returns text, label, and encodings
    def __getitem__(self, idx):

        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(text, padding='max_length', max_length=128, truncation=True,return_tensors='pt')

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [25]:
#Creates training and evaluation SentimentDataset objects
train_dataset = SentimentDataset(x_train, y_train, tokenizer)

eval_dataset = SentimentDataset(x_test, y_test, tokenizer)

In [26]:
#Sets up training configuration (batch size, learning rate, etc.) and trainer model object
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,

    logging_steps=100,

    eval_strategy='epoch',
    save_strategy='epoch',

    report_to='none'
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=eval_dataset)

In [27]:
#Training
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.545900,0.524621
2,0.460179,0.479967
3,0.334264,0.527247


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4500, training_loss=0.47472254604763453, metrics={'train_runtime': 1975.9076, 'train_samples_per_second': 36.439, 'train_steps_per_second': 2.277, 'total_flos': 4736041519104000.0, 'train_loss': 0.47472254604763453, 'epoch': 3.0})

In [28]:
#Evaluation (Loss)
trainer.evaluate()

Training Loss,Validation Loss,Epoch
0.334264,0.527247,3


{'eval_loss': 0.5272474884986877}

In [29]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

#Runs the trained model on validation dataset
results = trainer.predict(eval_dataset)

#Converts model outputs into predicted class labels
y_pred = np.argmax(results.predictions, axis=1)

#Gets the actual labels
y_true = results.label_ids

In [31]:
#Prints Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[2131   31  448]
 [  89   42   82]
 [ 500   22 2655]]


In [32]:
#Prints Classification Report (Precision, recall, f-1 score, tec.)
print(classification_report(
    y_true,
    y_pred,
    target_names=['Negative', 'Neutral', 'Positive']
))

              precision    recall  f1-score   support

    Negative       0.78      0.82      0.80      2610
     Neutral       0.44      0.20      0.27       213
    Positive       0.83      0.84      0.83      3177

    accuracy                           0.80      6000
   macro avg       0.69      0.62      0.64      6000
weighted avg       0.80      0.80      0.80      6000

